# MeetMind: full BERT-base GPU training and pilot comparison

Enable a GPU: Runtime > Change runtime type > T4 GPU (or another CUDA GPU). Run cells in order. This notebook trains two full-data BERT-base candidates, selects using validation only, and compares against the existing tiny pilot. Final test data is never uploaded or evaluated. Actual results appear only after the GPU jobs finish; this notebook has not been executed by the coding agent.

Prepare `bert-base-training-inputs.zip` locally using the exporter described in backend/BERT_BASE_COMPARISON.md. Only public AMI train/validation data and public-data pilot predictions are transferred. No private meeting recordings are needed. Google Drive is used to retain completed epoch checkpoints across runtime disconnects.


In [ ]:
import sys, subprocess, pathlib, os, json, hashlib, zipfile, io
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing.'
print('GPU:', torch.cuda.get_device_name(0), 'PyTorch:', torch.__version__)
# Keep the runtime's CUDA-enabled PyTorch; do not install the project's CPU uv source.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers==4.57.6', 'scikit-learn==1.9.1', 'safetensors'])
REPO_URL = 'https://github.com/AhmadHassanSE/Meet_Mind.git'
BRANCH = 'feat/bert-base-comparison'
repo = pathlib.Path('/content/Meet_Mind')
if not repo.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(repo)])
backend = repo / 'meetmind' / 'backend'
os.chdir(backend)
source_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Training source revision:', source_revision)


In [ ]:
from google.colab import drive, files
PERSIST_TO_DRIVE = True
EXPERIMENT_NAME = 'bert-base-comparison-v1'
if PERSIST_TO_DRIVE:
    drive.mount('/content/drive')
    experiment = pathlib.Path('/content/drive/MyDrive/MeetMind-BERT') / EXPERIMENT_NAME
else:
    experiment = pathlib.Path('/content/MeetMind-BERT') / EXPERIMENT_NAME
experiment.mkdir(parents=True, exist_ok=True)
print('Persistent experiment directory:', experiment)
# Colab prompts you to choose the locally generated input ZIP.
uploaded = files.upload()
assert len(uploaded) == 1, 'Choose only bert-base-training-inputs.zip.'
archive_bytes = next(iter(uploaded.values()))
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    expected = {'data/ami-da-v1/train.jsonl', 'data/ami-da-v1/validation.jsonl', 'data/ami-da-v1/manifest.json', 'reference/pilot-validation.json', 'input-manifest.json'}
    assert set(archive.namelist()) == expected, 'Unexpected ZIP contents; final test must be excluded.'
    inputs = json.loads(archive.read('input-manifest.json'))
    assert inputs['test_data_included'] is False
    for name, digest in inputs['files'].items():
        data = archive.read(name)
        assert hashlib.sha256(data).hexdigest() == digest, name
        target = backend / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert target.read_bytes() == data, 'Existing dataset differs: use a fresh runtime.'
        else:
            target.write_bytes(data)
assert not pathlib.Path('data/ami-da-v1/test.jsonl').exists()
(experiment / 'input-manifest.json').write_text(json.dumps(inputs, indent=2))
(experiment / 'source-revision.txt').write_text(source_revision + '\n')
print('Verified byte-identical public train/validation inputs; final test absent.')


In [ ]:
def run_module(module, *args):
    subprocess.check_call([sys.executable, '-u', '-m', module, *map(str, args)])
# A short real BERT-base GPU training check before committing to the full run.
# Its scores are NOT model-selection evidence.
smoke = json.loads(pathlib.Path('configs/bert-base-gpu-lr2e5.json').read_text())
smoke.update(output=str(experiment / 'smoke'), stage='smoke-test', max_train_examples=256,
             max_validation_examples=128, epochs=1, save_epoch_state=False)
smoke_config = experiment / 'smoke-config.json'
smoke_config.write_text(json.dumps(smoke, indent=2))
smoke_status = experiment / 'smoke/status.json'
if not smoke_status.exists() or json.loads(smoke_status.read_text())['status'] != 'completed':
    run_module('scripts.train_bert', '--config', smoke_config)
else:
    print('Completed smoke retained.')


In [ ]:
configs = []
for suffix in ['lr2e5', 'lr3e5']:
    cfg = json.loads(pathlib.Path(f'configs/bert-base-gpu-{suffix}.json').read_text())
    cfg['output'] = str(experiment / f'bert-base-gpu-{suffix}')
    path = experiment / f'{suffix}.json'
    if path.exists():
        assert json.loads(path.read_text()) == cfg, 'Experiment config changed; choose a new experiment name.'
    else:
        path.write_text(json.dumps(cfg, indent=2))
    configs.append(path)
comparison_dir = experiment / 'comparison'
if not (comparison_dir / 'comparison.json').exists():
    run_module('scripts.run_bert_comparison', '--configs', *configs,
               '--reference', 'reference/pilot-validation.json', '--output', comparison_dir, '--resume')
else:
    print('Completed comparison retained.')
print((comparison_dir / 'comparison.md').read_text())


In [ ]:
# Export trained full-size checkpoints and compact evidence without changing the app.
# The pilot remains the serving model until these actual results are reviewed.
from nlp.data import write_json
for config in configs:
    cfg = json.loads(config.read_text())
    run = pathlib.Path(cfg['output'])
    evidence = experiment / 'reports' / run.name
    if not evidence.exists():
        run_module('scripts.summarize_bert', '--run', run, '--output', evidence)
summary = json.loads((comparison_dir / 'comparison.json').read_text())
print('Selected validation report:', summary['selected_report'])
print('Macro F1 gain over tiny pilot:', summary['selected_macro_f1_gain'])
print('Full checkpoint folders are in', experiment)
# Include only best model weights and compact public-data evidence, not optimizer states or corpus files.
export_path = experiment / 'trained-bert-base-results.zip'
if not export_path.exists():
    with zipfile.ZipFile(export_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for root in [experiment / 'comparison', experiment / 'reports'] + [pathlib.Path(json.loads(c.read_text())['output']) / 'best' for c in configs]:
            for path in root.rglob('*'):
                if path.is_file():
                    archive.write(path, path.relative_to(experiment))
        archive.writestr('source-revision.txt', source_revision + '\n')
print('Saved trained models/evidence ZIP:', export_path)
print('Copy the ZIP from Drive to your local machine after reviewing the comparison.')


## If Colab disconnects

Reconnect a GPU runtime and rerun the cells with the same `EXPERIMENT_NAME` and input ZIP. Training resumes from the latest **completed epoch**, not the interrupted mini-batch. Keep the same source revision and configuration. Exact reproducibility across different GPU architectures/software versions is not guaranteed.

The two candidates use all training rows; the old tiny pilot used 24,000 rows and a shorter input length. Improvements are therefore comparisons of complete candidate setups, not proof that architecture alone caused a gain. No final-test performance is claimed. Do not select a smoke model or replace the existing serving model automatically.
